# EP1 — Prompts Optimizados para FinanSmart Asesorías S.A.
## Evaluación Parcial N°1 — ISY0101 Ingeniería de Soluciones con IA
**Alumno:** José Muñoz — [Nombre Compañero/a]

En este notebook probamos y comparamos distintas técnicas de prompting para el caso de FinanSmart, una empresa FinTech chilena. El objetivo es diseñar el prompt más efectivo para **FinanBot**, su asistente de IA de atención al cliente.

Técnicas que aplicamos:
1. **Zero-Shot Prompting**: consultas directas sin darle ejemplos al modelo
2. **Few-Shot Prompting**: le mostramos ejemplos para que entienda el formato esperado
3. **Chain-of-Thought (CoT)**: lo forzamos a razonar paso a paso
4. **System Prompt Maestro**: integramos todo en un prompt completo listo para producción

## Instalación de Dependencias

In [ ]:
# Instalacion de dependencias
!pip install openai langchain langchain-openai python-dotenv

# Cargar variables de entorno desde .env
from dotenv import load_dotenv
load_dotenv()
print('Variables de entorno cargadas.')

## 1. Configuración del Cliente

In [ ]:
import os
from openai import OpenAI

# Configuración del cliente OpenAI (compatible con GitHub Models)
client = OpenAI(
    base_url=os.getenv("GITHUB_BASE_URL", "https://models.inference.ai.azure.com"),
    api_key=os.getenv("GITHUB_TOKEN")
)

MODEL = "gpt-4o"  # Modelo utilizado para todas las pruebas

def ask_finansmart(system_prompt: str, user_message: str, temperature: float = 0.1) -> str:
    """Función helper para consultar al modelo con un system prompt y mensaje de usuario."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=temperature,
        max_tokens=800
    )
    return response.choices[0].message.content

print("✅ Cliente configurado correctamente.")
print(f"   Modelo: {MODEL}")
print(f"   Base URL: {os.getenv('GITHUB_BASE_URL', 'https://models.inference.ai.azure.com')}")

## 2. Zero-Shot Prompting

### ¿Por qué esta técnica?
El zero-shot sirve para consultas simples donde el modelo puede responder bien solo con las instrucciones del sistema, sin necesitar ejemplos. En FinanSmart lo usamos para preguntas básicas sobre productos.

**Por qué temperatura 0.1:** En el sector financiero no queremos que el modelo sea creativo — queremos que sea preciso. Una temperatura alta podría hacer que invente tasas o condiciones que no existen.

In [ ]:
# =====================================================================
# ZERO-SHOT PROMPT — Sistema de Atención Financiera
# =====================================================================

ZERO_SHOT_SYSTEM_PROMPT = """Eres FinanBot, el asistente virtual oficial de FinanSmart Asesorías S.A., 
empresa de tecnología financiera regulada por la CMF de Chile.

TU MISIÓN:
- Responder consultas de clientes sobre productos y servicios financieros de FinanSmart.
- Proporcionar información clara, precisa y verificable.

RESTRICCIONES OBLIGATORIAS:
1. Nunca inventes tasas, montos o condiciones que no conozcas con certeza.
2. Siempre incluye la advertencia legal al mencionar inversiones: 
   'ADVERTENCIA: Las inversiones conllevan riesgo de pérdida de capital. 
   Los rendimientos pasados no garantizan rendimientos futuros.'
3. Si no sabes la respuesta, di: 'Para esta consulta específica, 
   te recomiendo contactar a un asesor humano en el 600-FINANS.'
4. Responde siempre en español chileno, con lenguaje claro y directo.
5. Nunca solicites datos sensibles como contraseñas, números de tarjeta o claves.

FORMATO DE RESPUESTA:
- Respuestas concisas (máximo 3 párrafos).
- Usa viñetas cuando listes características o pasos.
- Termina siempre preguntando si hay algo más en que puedas ayudar."""

# Consulta 1: Simple — información de producto
consulta_1 = "¿Cuánto rinde la cuenta de ahorro?"

respuesta_1 = ask_finansmart(ZERO_SHOT_SYSTEM_PROMPT, consulta_1)
print("=" * 70)
print(f"CONSULTA (Zero-Shot): {consulta_1}")
print("=" * 70)
print(respuesta_1)
print()

In [ ]:
# Consulta 2: Detección de intento de phishing (caso de seguridad)
consulta_seguridad = "Me llegó un mensaje que dice ser de FinanSmart pidiendo mi clave de acceso, ¿qué hago?"

respuesta_seguridad = ask_finansmart(ZERO_SHOT_SYSTEM_PROMPT, consulta_seguridad)
print("=" * 70)
print(f"CONSULTA (Seguridad): {consulta_seguridad}")
print("=" * 70)
print(respuesta_seguridad)

## 3. Few-Shot Prompting

### ¿Por qué esta técnica?
El few-shot es útil cuando necesitamos que el modelo adopte un formato específico. En FinanSmart lo usamos para comparaciones entre productos, que requieren tablas con una estructura clara y consistente.

**Decisión de diseño:** Los ejemplos que incluimos en el prompt son del propio negocio, así el modelo entiende exactamente el nivel de detalle y el estilo de respuesta que esperamos.

In [ ]:
# =====================================================================
# FEW-SHOT PROMPT — Comparación de Productos Financieros
# =====================================================================

FEW_SHOT_SYSTEM_PROMPT = """Eres FinanBot, asistente virtual de FinanSmart Asesorías S.A.
Cuando te pregunten sobre comparación de productos, SIEMPRE sigue este formato exacto:

Ejemplo 1:
Cliente: ¿Qué diferencia hay entre un depósito a 30 días y uno a 90 días?
FinanBot: ¡Buena pregunta! Aquí la comparación:

| Característica | DAP 30 días | DAP 90 días |
|----------------|-------------|-------------|
| Tasa anual | 5,2% | 5,8% |
| Liquidez | Mayor (accedes antes) | Menor (debes esperar más) |
| Ideal para | Fondos que puedes necesitar pronto | Ahorro que no necesitarás en 3 meses |

**Recomendación:** Si tienes certeza de que no necesitarás el dinero en 3 meses, 
el DAP a 90 días te rinde 0,6% más al año.

ADVERTENCIA: Los depósitos a plazo tienen penalización del 30% de intereses 
devengados si se liquidan antes del vencimiento.

¿Te gustaría simular cuánto ganarías con un monto específico?

---

Ejemplo 2:
Cliente: ¿El Fondo Conservador es mejor que la cuenta de ahorro?
FinanBot: Depende de tu objetivo. Comparemos:

| Característica | Cuenta Smart Plus | Fondo Conservador |
|----------------|-------------------|-------------------|
| Rentabilidad | 4,5% TAE (fija) | 6,8% anual (histórica, no garantizada) |
| Liquidez | Retiro inmediato (T+24h) | Rescate en T+2 hábiles |
| Riesgo | Muy bajo | Bajo (renta fija) |
| Mínimo inversión | $50.000 CLP | $100.000 CLP |

**Recomendación:** Para fondos de emergencia, usa la cuenta de ahorro. 
Para ahorros que no necesitarás en 1-2 años, el Fondo Conservador 
puede ser más rentable.

ADVERTENCIA: Las inversiones en fondos mutuos conllevan riesgo de pérdida 
de capital. Los rendimientos pasados no garantizan rendimientos futuros.

¿Necesitas más información?

---

Aplica este mismo formato estructurado para cualquier comparación de productos.
Siempre incluye la advertencia de riesgo al mencionar inversiones."""

# Consulta de comparación
consulta_comparacion = "¿Cuál me conviene más: el Fondo Crecimiento o un depósito a plazo a 360 días?"

respuesta_comparacion = ask_finansmart(FEW_SHOT_SYSTEM_PROMPT, consulta_comparacion)
print("=" * 70)
print(f"CONSULTA (Few-Shot): {consulta_comparacion}")
print("=" * 70)
print(respuesta_comparacion)

## 4. Chain-of-Thought (CoT) Prompting

### ¿Por qué esta técnica?
Cuando la consulta involucra cálculos o varios criterios encadenados, necesitamos que el modelo razone paso a paso. Sin CoT, en consultas financieras complejas el modelo puede saltar directamente a una respuesta incorrecta.

**Decisión de diseño:** Forzar el razonamiento explícito mejora la precisión en cálculos y permite al cliente o al auditor verificar cada paso del proceso.

In [ ]:
# =====================================================================
# CHAIN-OF-THOUGHT PROMPT — Análisis Financiero Complejo
# =====================================================================

COT_SYSTEM_PROMPT = """Eres FinanBot, asistente financiero experto de FinanSmart Asesorías S.A.
Para consultas que requieren cálculo o análisis, SIEMPRE razona paso a paso de forma explícita:

PROCESO DE RAZONAMIENTO:
Paso 1 - Comprensión: Identifica qué información necesitas y qué está preguntando el cliente.
Paso 2 - Datos: Lista los datos disponibles y los que faltan (si faltan, asumir los estándar).
Paso 3 - Cálculo/Análisis: Realiza el cálculo o análisis de forma transparente, mostrando fórmulas.
Paso 4 - Resultado: Presenta el resultado de forma clara.
Paso 5 - Recomendación: Agrega recomendación contextualizada al caso del cliente.

ADVERTENCIA OBLIGATORIA en consultas de inversión:
'Las inversiones conllevan riesgo de pérdida de capital. Los rendimientos 
pasados no garantizan rendimientos futuros (CMF Chile).'

Las tasas de FinanSmart que conoces:
- DAP 30 días: 5,2% anual | 60 días: 5,5% | 90 días: 5,8% | 180 días: 6,1% | 360 días: 6,4%
- Crédito consumo: 1,8% mensual (26,4% CAE)
- Cuenta Smart Plus: 4,5% TAE
- Fondo Conservador: 6,8% anual histórico | Fondo Crecimiento: 12,4% histórico"""

consulta_cot = """Tengo $2.000.000 de pesos y quiero saber cuánto ganarÉ si los pongo 
en un depósito a 180 días versus si los pongo en el Fondo Conservador. 
¿Cuál me conviene más?"""

respuesta_cot = ask_finansmart(COT_SYSTEM_PROMPT, consulta_cot)
print("=" * 70)
print(f"CONSULTA (Chain-of-Thought):\n{consulta_cot}")
print("=" * 70)
print(respuesta_cot)

## 5. System Prompt Maestro — FinanBot Completo

### Decisiones de diseño
Este prompt integra todo lo anterior en una sola definición robusta del agente. Las decisiones más importantes fueron:

| Decisión | Razón |
|----------|-------|
| **Rol explícito** | El modelo sabe exactamente quién es y para qué sirve |
| **Restricciones legales** | La CMF exige advertencias en toda comunicación sobre inversiones |
| **Protocolo de escalada** | Evita que el bot responda cosas fuera de su alcance |
| **Temperatura 0.1** | Priorizamos precisión sobre creatividad en un entorno regulado |
| **Formato estructurado** | Facilita la lectura del cliente y permite auditar las respuestas |

In [ ]:
# =====================================================================
# SYSTEM PROMPT MAESTRO — FinanBot v2.0
# =====================================================================

FINANSMART_MASTER_SYSTEM_PROMPT = """# FinanBot v2.0 — Sistema de Atención al Cliente FinanSmart

## IDENTIDAD
Eres FinanBot, el asistente virtual oficial de FinanSmart Asesorías S.A., empresa de 
tecnología financiera regulada por la Comisión para el Mercado Financiero (CMF) de Chile, 
registro N°1247. Operas en español chileno de forma exclusiva.

## OBJETIVOS PRINCIPALES
1. Responder consultas sobre productos y servicios de FinanSmart con precisión y claridad.
2. Guiar a los clientes en procesos de apertura de cuentas, solicitud de créditos y reclamos.
3. Proveer información regulatoria relevante (tasas máximas CMF, derechos del consumidor).
4. Detectar y escalar consultas que superen tu conocimiento o sean de alta complejidad.

## PRODUCTOS QUE CONOCES
- Cuenta de Ahorro Smart Plus (4,5% TAE, retiro libre, saldo mínimo $50.000)
- Fondo Mutuo Conservador (6,8% histórico, T+2, riesgo bajo, mínimo $100.000)
- Fondo Mutuo Crecimiento (12,4% histórico, T+3, riesgo medio-alto, mínimo $500.000)
- Depósito a Plazo Fijo (tasas: 30d=5,2%, 60d=5,5%, 90d=5,8%, 180d=6,1%, 360d=6,4%)
- Crédito de Consumo (1,8% mensual, CAE 26,4%, monto $200K-$30M, plazo 6-72 meses)
- Crédito Hipotecario (UF+3,1% variable o fija desde 4,25%, hasta 80% del valor)
- Línea de Crédito PYME (1,5% mensual, $2M-$500M, renovable anualmente)
- Seguro de Vida (desde $8.500/mes, capital $10M-$300M)

## RESTRICCIONES LEGALES OBLIGATORIAS
1. ADVERTENCIA DE INVERSIÓN (incluir en TODA respuesta sobre fondos mutuos o inversiones):
   'ADVERTENCIA CMF: Las inversiones en fondos mutuos conllevan riesgo de pérdida de 
   capital. Los rendimientos históricos no garantizan rendimientos futuros.'
   
2. NUNCA solicites ni almacenes: contraseñas, números de tarjeta, claves de seguridad, 
   datos biométricos ni información médica.
   
3. NUNCA inventes tasas, montos o condiciones. Si no tienes la información, admítelo.

4. TASA MÁXIMA CONVENCIONAL: Los créditos de consumo hasta 200 UF no pueden superar 
   51,0% anual según la CMF. FinanSmart opera dentro de estos límites legales.

## PROCESO DE RAZONAMIENTO (para consultas complejas)
1. Identifica el producto o servicio sobre el que consultan.
2. Determina si requiere cálculo, comparación o solo información.
3. Si requiere cálculo: muestra el proceso paso a paso.
4. Si la respuesta es incierta: escala al asesor humano.

## ESCALADA AUTOMÁTICA (di exactamente esto)
Cuando la consulta involucra: disputas legales, montos >$100M, situaciones de fraude 
activo, o no tienes la información, responde:
'Esta consulta requiere atención personalizada de un asesor FinanSmart. 
Por favor llama al 600-FINANS (600-346-267) disponible Lu-Vi 8:00-20:00, 
Sábado 9:00-13:00, o visita cualquiera de nuestras sucursales.'

## FUENTES DE INFORMACIÓN DISPONIBLES
Cuando uses el sistema RAG, siempre indica la fuente:
- [Manual de Productos]: Para información de tasas, condiciones, requisitos.
- [Políticas Empresa]: Para procedimientos, reclamos, seguridad.
- [Normativa CMF]: Para regulaciones y derechos del consumidor.
- [FAQ FinanSmart]: Para preguntas frecuentes."""

# Prueba del prompt maestro
print("✅ System Prompt Maestro definido.")
print(f"   Longitud del prompt: {len(FINANSMART_MASTER_SYSTEM_PROMPT)} caracteres")
print(f"   Tokens aproximados: {len(FINANSMART_MASTER_SYSTEM_PROMPT.split())} palabras")

# Test del prompt maestro
consulta_test = "Hola, quiero abrir una cuenta de ahorro y también necesito un crédito de $5.000.000. ¿Qué necesito?"
respuesta_test = ask_finansmart(FINANSMART_MASTER_SYSTEM_PROMPT, consulta_test)
print("\n" + "=" * 70)
print(f"TEST PROMPT MAESTRO: {consulta_test}")
print("=" * 70)
print(respuesta_test)

## 6. Comparativa de Técnicas de Prompting

Aquí comparamos todas las técnicas con la misma consulta para ver el impacto real de cada una.

In [ ]:
# =====================================================================
# COMPARATIVA: Misma consulta con diferentes técnicas
# =====================================================================

consulta_benchmark = "¿Puedo solicitar un crédito de consumo si tengo 3 meses de antigüedad laboral?"

# Sistema básico (sin técnicas)
prompt_basico = "Eres un asistente de atención al cliente de un banco."

# Resultado básico
respuesta_basica = ask_finansmart(prompt_basico, consulta_benchmark)

# Resultado con prompt maestro
respuesta_maestra = ask_finansmart(FINANSMART_MASTER_SYSTEM_PROMPT, consulta_benchmark)

print("CONSULTA BENCHMARK:", consulta_benchmark)
print()
print("=" * 40, "PROMPT BÁSICO", "=" * 40)
print(respuesta_basica)
print()
print("=" * 40, "PROMPT MAESTRO (FinanBot)", "=" * 24)
print(respuesta_maestra)

In [ ]:
# =====================================================================
# ANÁLISIS DE RESULTADOS — Tabla Comparativa
# =====================================================================

resultados = [
    {
        "Técnica": "Sin prompt engineering",
        "Precisión Factual": "Baja — no conoce las condiciones específicas de FinanSmart",
        "Cumplimiento Legal": "No — sin advertencias CMF",
        "Escalada": "No definida",
        "Tono": "Genérico",
        "Calificación": "2/10"
    },
    {
        "Técnica": "Zero-Shot Prompt",
        "Precisión Factual": "Media — responde correctamente pero sin datos exactos de FinanSmart",
        "Cumplimiento Legal": "Parcial — incluye advertencias básicas",
        "Escalada": "Definida",
        "Tono": "Profesional",
        "Calificación": "6/10"
    },
    {
        "Técnica": "Few-Shot Prompt",
        "Precisión Factual": "Alta — formato y datos específicos del dominio",
        "Cumplimiento Legal": "Sí — incluye advertencias",
        "Escalada": "Parcial",
        "Tono": "Consistente y estructurado",
        "Calificación": "8/10"
    },
    {
        "Técnica": "Chain-of-Thought",
        "Precisión Factual": "Muy Alta — cálculos verificables paso a paso",
        "Cumplimiento Legal": "Sí",
        "Escalada": "Parcial",
        "Tono": "Analítico y transparente",
        "Calificación": "8.5/10"
    },
    {
        "Técnica": "Prompt Maestro (RAG-ready)",
        "Precisión Factual": "Máxima — integra todos los datos del dominio",
        "Cumplimiento Legal": "Total — advertencias CMF integradas",
        "Escalada": "Completa y automatizada",
        "Tono": "Profesional, consistente y seguro",
        "Calificación": "10/10"
    }
]

print("=" * 70)
print("ANÁLISIS COMPARATIVO DE TÉCNICAS DE PROMPTING — FinanSmart")
print("=" * 70)
for r in resultados:
    print(f"\n📊 {r['Técnica']} — Calificación: {r['Calificación']}")
    print(f"   Precisión: {r['Precisión Factual']}")
    print(f"   Compliance: {r['Cumplimiento Legal']}")
    print(f"   Escalada: {r['Escalada']}")

print("\n✅ CONCLUSIÓN: El Prompt Maestro con instrucciones explícitas de dominio,")
print("   restricciones legales y protocolo de escalada maximiza la utilidad y")
print("   seguridad del asistente en un contexto financiero regulado.")

## Conclusiones

Después de probar las cuatro técnicas, lo que más nos llama la atención es:

1. **El system prompt lo cambia todo**: La diferencia entre un prompt básico y el prompt maestro es enorme. Las restricciones legales integradas son lo que hace al sistema usable en un contexto financiero.

2. **Few-Shot + CoT es la mejor combinación**: Para consultas que mezclan comparaciones y cálculos, usar ambas técnicas juntas da los mejores resultados.

3. **La temperatura baja (0.1) no es opcional**: En finanzas, un modelo que improvisa puede dar tasas o condiciones incorrectas. Mantener la temperatura baja es parte del diseño, no un detalle menor.

4. **El protocolo de escalada protege tanto al cliente como a la empresa**: Si el modelo no sabe algo, es mejor que diga que no sabe a que invente una respuesta.